# Incremental Load in Bronze

In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    lit,
    sha2,
    concat_ws,
    col,
    regexp_extract,
    to_date
)

# -----------------------------------
# CONFIG
# -----------------------------------
base_path = "/Volumes/retail/default/raw_parquet"
run_id = "RUN_002"
source_system = "Postgres"

table_configs = [
    {
        "table_name": "ns_brands",
        "target_table": "retail.bronze.ns_brands",
        "hash_cols": ["internal_id", "brand_code", "brand_name", "is_active", "created_at"]
    },
    {
        "table_name": "ns_customers",
        "target_table": "retail.bronze.ns_customers",
        "hash_cols": [
            "internal_id", "entity_id", "company_name", "customer_type", "email", "phone",
            "subsidiary", "currency_code", "terms_name", "vat_number",
            "billing_address1", "billing_city", "billing_state", "billing_country",
            "shipping_address1", "shipping_city", "shipping_state", "shipping_country",
            "is_active", "created_at"
        ]
    },
    {
        "table_name": "ns_items",
        "target_table": "retail.bronze.ns_items",
        "hash_cols": [
            "internal_id", "item_id", "item_name", "item_type", "category", "subcategory",
            "brand_internal_id", "base_price", "cost_price", "is_active", "created_at"
        ]
    },
    {
        "table_name": "ns_sales_orders",
        "target_table": "retail.bronze.ns_sales_orders",
        "hash_cols": [
            "internal_id", "tran_id", "customer_internal_id", "order_date", "order_status",
            "sales_channel", "payment_status", "currency_code", "external_ref_num",
            "integration_source", "order_total", "created_at"
        ]
    },
    {
        "table_name": "ns_sales_order_lines",
        "target_table": "retail.bronze.ns_sales_order_lines",
        "hash_cols": [
            "internal_id", "sales_order_internal_id", "line_num", "item_internal_id",
            "quantity", "unit_rate", "line_amount", "created_at"
        ]
    },
    {
        "table_name": "ns_shipments",
        "target_table": "retail.bronze.ns_shipments",
        "hash_cols": [
            "internal_id", "shipment_number", "sales_order_internal_id", "warehouse_name",
            "shipment_date", "delivery_date", "shipment_status", "tracking_number", "created_at"
        ]
    }
]

# -----------------------------------
# FUNCTION
# -----------------------------------
def load_incremental(table_name, target_table, hash_cols):
    path = f"{base_path}/{table_name}_*.parquet"

    print(f"Processing {table_name} from {path}")

    # Read parquet + metadata
    df = (
        spark.read
        .format("parquet")
        .load(path)
        .select("*", "_metadata")
        .withColumn("source_file_path", col("_metadata.file_path"))
        .drop("_metadata")
    )

    # Cast business columns to string
    business_cols = [c for c in df.columns if c != "source_file_path"]

    for c in business_cols:
        df = df.withColumn(c, col(c).cast("string"))

    # Build record hash from source/business columns only
    hash_expr = concat_ws("||", *[col(c) for c in hash_cols])

    # Add Bronze metadata
    df_final = (
        df.withColumn("etl_loaded_at", current_timestamp())
          .withColumn("source_system", lit(source_system))
          .withColumn("etl_run_id", lit(run_id))
          .withColumn(
              "ingestion_date",
              to_date(
                  regexp_extract(col("source_file_path"), r"(\d{4}-\d{2}-\d{2})", 1),
                  "yyyy-MM-dd"
              )
          )
          .withColumn("record_hash", sha2(hash_expr, 256))
          .withColumn("is_deleted", lit(False))
          .drop("source_file_path")
    )

    # Append into Bronze
    df_final.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(target_table)

    print(f"{table_name} incremental load done")

# -----------------------------------
# RUN
# -----------------------------------
for cfg in table_configs:
    load_incremental(
        cfg["table_name"],
        cfg["target_table"],
        cfg["hash_cols"]
    )